In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, load_tools   
from langchain.tools import StructuredTool, Tool

llm = ChatOpenAI(temperature=0.1)


### STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION

Input을 여러 개 받을 수 있다. (임의로 지정할 수 있다.)
아래의 agent invoke 시 agent 내에 기재한 함수 (StructuredTool.from_function->plus가 재귀적으로 도는 것을 볼 수 있다.)

In [10]:

def plus(a, b):
    return a+b

agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    tools=[
    StructuredTool.from_function(
        func=plus,
        name="Sum Calculator",
        description="Use this to perform sums of two numbers. This tool take two arguments, both should be numbers"
    )    
])

prompt = "Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12"

agent.invoke(prompt)



> Entering new AgentExecutor chain...
Thought:
I can use the Sum Calculator tool to find the total cost of all these amounts.

Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 355.39, "b": 924.87}
}
```
Observation: 1280.26
Thought:I have calculated the sum of the first two amounts. Now, I will continue to calculate the sum of all the remaining amounts.
Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 1280.26, "b": 721.2}
}
```
Observation: 2001.46
Thought:I have calculated the sum of the first two amounts and the sum of the next two amounts. Now, I will continue to calculate the sum of all the remaining amounts.

Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 2001.46, "b": 1940.29}
}
```
Observation: 3941.75
Thought:I have calculated the sum of the first four amounts. Now, I will continue to calculate the sum of all the remaining amounts.

Action:
```
{
  "action": "Sum Calculator",
  "action_input": {"a": 3941.75, "b": 573.

{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost is $4581.10.'}

### AgentExecutor
```python
next_action = agent.get_action(...)
while next_action != AgentFinish:
    observation = run(next_action)
    next_action = agent.get_action(..., next_action, observation)
return next_action
```

--- 아래는 langsmith를 통해 확인한 prompt 내용 ---

Respond to the human as helpfully and accurately as possible. You have access to the following tools:
Sum Calculator: Sum Calculator(a, b) - Use this to perform sums of two numbers. This tool take two arguments, both should be numbers, args: {{'a': {{'title': 'A'}}, 'b': {{'title': 'B'}}}}
Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).
Valid "action" values: "Final Answer" or Sum Calculator
Provide only ONE action per $JSON_BLOB, as shown:

```
{
  "action": $TOOL_NAME,
  "action_input": $INPUT
}
```

Follow this format:

Question: input question to answer
Thought: consider previous and subsequent steps
Action:
```
$JSON_BLOB
```
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
```
{
  "action": "Final Answer",
  "action_input": "Final response to human"
}
```

Begin! Reminder to ALWAYS respond with a valid json blob of a single action. Use tools if necessary. Respond directly if appropriate. Format is Action:```$JSON_BLOB```then Observation:.
Thought:

### ZERO_SHOT_REACT_DESCRIPTION

Single input만 가능하다. 따라서 description에 어떤 형태의 input이 들어올 지 명시해야 하며,
Tool에 붙어 있는 함수에 별도 parse 기능을 추가로 구현해야 한다.

In [22]:
## Zero-shot Agent
## Only possible a single Input

def plus_beta(input):
    a, b = input.split(",")
    return float(a) + float(b)


In [ ]:


agent_beta = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    tools=[
    StructuredTool.from_function(
        func=plus_beta,
        name="Sum Calculator_beta",
        description="Use this to perform sums of two numbers. Use this tool by sending a pair of number separated by a comma. \nExample: 1, 2"
    )    
])

agent_beta.invoke(prompt)

In [21]:
from pydantic import BaseModel, Field
from langchain.tools import BaseTool
from typing import Any, Type
## 데이터가 어떤 형태인지 알려 주는 라이브러리 (User 등의 데이터가 일반적으로 어떤 형태의 구조를 가지는지 미리 정의)
## OpenAI Function

## 구현이 매우매우 효율이 안 좋아보였는데, 프롬프트 내용을 단 한줄로 둘 수 있다.
## 이후 여러 Tool을 구현해 보면서는 Schema를 재활용 가능하다는 점에서 유연성이 더 뛰어난 것으로 보임.
## 하나의 Schema(Input)을 통해 각기 다른 기능을 하는 Tool들을 구현해 agent에 붙이는 형태로 구현 (주로 이 방법을 쓰는 것으로 보임)

class CalculatorToolArgsSchema(BaseModel):
    a: float = Field(description="The first number")
    b: float = Field(description="The second number")

class CalculatorTool(BaseTool):
    name = "CalculatorTool"
    description = "Use this to perform sums of two numbers."
    args_schema : Type[CalculatorToolArgsSchema] = CalculatorToolArgsSchema

    def _run(self, a, b):
        return a+b


agent_gamma = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[CalculatorTool()]
)

agent_gamma.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `CalculatorTool` with `{'a': 355.39, 'b': 924.87}`


1280.26
Invoking: `CalculatorTool` with `{'a': 1280.26, 'b': 721.2}`


2001.46
Invoking: `CalculatorTool` with `{'a': 2001.46, 'b': 1940.29}`


3941.75
Invoking: `CalculatorTool` with `{'a': 3941.75, 'b': 573.63}`


4515.38
Invoking: `CalculatorTool` with `{'a': 4515.38, 'b': 65.72}`


4581.1
Invoking: `CalculatorTool` with `{'a': 4581.1, 'b': 35.0}`


4616.1
Invoking: `CalculatorTool` with `{'a': 4616.1, 'b': 552.0}`


5168.1
Invoking: `CalculatorTool` with `{'a': 5168.1, 'b': 76.16}`


5244.26
Invoking: `CalculatorTool` with `{'a': 5244.26, 'b': 29.12}`


5273.38The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.

> Finished chain.


{'input': 'Cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12',
 'output': 'The total cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $552.00 + $76.16 + $29.12 is $5273.38.'}